# Stage 6 — Reranking: Baseline vs. MMR vs. Hybrid (Vector + BM25)

Input: `data/eval_set.jsonl` (Stage 5's 20 questions with known-correct chunks) + `data/chunks/chunks.jsonl`.

Stage 5 showed HNSW and IVF_FLAT match FLAT's accuracy exactly while being ~2.5x faster — so this notebook **stops comparing three indexes and picks one to carry forward** (HNSW).

Three experiments, compared on the same eval set and the same metrics:

1. **Baseline** — plain vector search top-5, no reranking
2. **MMR** — vector search top-15 candidates, reranked down to top-5 for relevance + diversity
3. **Hybrid** — vector search top-15 **and** BM25 (lexical/keyword) top-15, fused via Reciprocal Rank Fusion (RRF) into a final top-5

Metrics for all three: Recall@5, average pairwise similarity within the selected set (lower = more diverse), and how many distinct chunk types appear.

Requires Milvus running and the same Azure OpenAI `.env` vars as Stage 5.


In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from pymilvus import connections, Collection
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CHUNKS_PATH = DATA_DIR / "chunks" / "chunks.jsonl"
EVAL_SET_PATH = DATA_DIR / "eval_set.jsonl"
LOGS_DIR = DATA_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

connections.connect(alias="default", host="localhost", port="19530")

CHOSEN_INDEX = "hnsw"
SEARCH_CONFIG = {"metric_type": "COSINE", "search_params": {"ef": 64}}
CANDIDATE_POOL_SIZE = 15
FINAL_K = 5
MMR_LAMBDA = 0.6

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [json.loads(line) for line in f]
chunks_by_id = {c["chunk_id"]: c for c in all_chunks}

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    eval_set = [json.loads(line) for line in f]

print(f"loaded {len(all_chunks)} chunks, {len(eval_set)} eval questions")

loaded 1301 chunks, 20 eval questions


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_44180/3327164914.py:21: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias="default", host="localhost", port="19530")


In [3]:
collection = Collection(f"chunks_{CHOSEN_INDEX}")
collection.load()
print(f"chunks_{CHOSEN_INDEX} ready, rows = {collection.num_entities}")

chunks_hnsw ready, rows = 1301


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_44180/751778325.py:1: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(f"chunks_{CHOSEN_INDEX}")
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_44180/751778325.py:2: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_44180/751778325.py:3: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"chunks_{CHOSEN_INDEX} ready, rows = {collection.num_entities}")


## Build a BM25 index over all chunks

BM25 is lexical (keyword) search, not semantic — it needs its own index built directly from the corpus text, entirely separate from Milvus's vector index. `rank_bm25` (already in `requirements.txt`, unused until now) just needs each document pre-split into tokens; a simple regex word-splitter is enough here, no heavy NLP dependency needed.


In [4]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25_corpus_ids = [c["chunk_id"] for c in all_chunks]
tokenized_corpus = [tokenize(c["content"]) for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(all_chunks)} chunks")

BM25 index built over 1301 chunks


## Candidate retrieval

`get_candidates` embeds the query, searches for the top `CANDIDATE_POOL_SIZE` (15) candidates, and attaches each candidate's full embedding from `chunks_by_id` — needed for MMR's diversity comparisons, and simpler than asking Milvus to return the vector field itself. `score` is Milvus's own cosine similarity, reused directly as the "relevance to query" term in MMR rather than recomputing it.


In [5]:
def embed_query(text: str) -> list[float]:
    response = client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=text)
    return response.data[0].embedding


def get_candidates(query_text: str, top_k: int = CANDIDATE_POOL_SIZE) -> list[dict]:
    query_vector = embed_query(query_text)
    results = collection.search(
        data=[query_vector],
        anns_field="embedding",
        param=SEARCH_CONFIG,
        limit=top_k,
        output_fields=["chunk_id", "doc_id", "page_number", "chunk_type"],
    )[0]

    candidates = []
    for hit in results:
        chunk_id = hit.entity.get("chunk_id")
        candidates.append({
            "chunk_id": chunk_id,
            "score": hit.distance,
            "embedding": chunks_by_id[chunk_id]["embedding"],
            "chunk_type": hit.entity.get("chunk_type"),
            "doc_id": hit.entity.get("doc_id"),
            "page_number": hit.entity.get("page_number"),
        })
    return candidates

## MMR (Maximal Marginal Relevance)

Greedily builds the final set one chunk at a time: pick the most relevant candidate first, then repeatedly pick whichever remaining candidate best balances **relevance to the query** against **dissimilarity to chunks already selected** — a highly-relevant-but-near-duplicate candidate gets penalized in favor of a somewhat-less-relevant but more novel one.

`MMR_LAMBDA` controls the balance: closer to 1 = prioritize pure relevance (behaves like plain top-k), closer to 0 = prioritize pure diversity (ignores relevance almost entirely). `0.6` here leans slightly toward relevance.


In [6]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def mmr_rerank(candidates: list[dict], k: int = FINAL_K, lam: float = MMR_LAMBDA) -> list[dict]:
    selected = []
    remaining = candidates.copy()

    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda c: c["score"])
        else:
            def mmr_score(c):
                relevance = c["score"]
                diversity_penalty = max(cosine_similarity(c["embedding"], s["embedding"]) for s in selected)
                return lam * relevance - (1 - lam) * diversity_penalty

            best = max(remaining, key=mmr_score)

        selected.append(best)
        remaining.remove(best)

    return selected

## Diversity metric

Average pairwise cosine similarity across every pair of chunks in a selected set — this is the number that should visibly drop when comparing MMR's selection against the plain top-k, since a lower average pairwise similarity means the selected chunks are less redundant with each other.


In [7]:
def avg_pairwise_similarity(chunks: list[dict]) -> float:
    if len(chunks) < 2:
        return 0.0
    sims = [
        cosine_similarity(chunks[i]["embedding"], chunks[j]["embedding"])
        for i in range(len(chunks))
        for j in range(i + 1, len(chunks))
    ]
    return sum(sims) / len(sims)

## Hybrid ranking: Vector + BM25, fused via Reciprocal Rank Fusion (RRF)

Vector cosine similarity and BM25 scores live on completely different, incompatible scales — combining them with a weighted sum would require normalizing both first, adding a hyperparameter to tune.

**RRF sidesteps this by fusing on *rank position* instead of raw score**: a chunk ranked #1 by either method contributes `1/(RRF_K + 1)`, ranked #2 contributes `1/(RRF_K + 2)`, and so on; a chunk appearing in both lists gets both contributions added together.

`RRF_K = 60` is the standard default used in most production hybrid-search implementations — it just softens how much rank 1 dominates over rank 2, 3, etc.


In [8]:
RRF_K = 60


def get_bm25_candidates(query_text: str, top_k: int = CANDIDATE_POOL_SIZE) -> list[str]:
    scores = bm25.get_scores(tokenize(query_text))
    ranked_idx = np.argsort(scores)[::-1][:top_k]
    return [bm25_corpus_ids[i] for i in ranked_idx]


def rrf_fuse(vector_candidates: list[dict], bm25_chunk_ids: list[str], final_k: int = FINAL_K) -> list[dict]:
    rrf_scores = {}

    for rank, c in enumerate(vector_candidates, start=1):
        rrf_scores[c["chunk_id"]] = rrf_scores.get(c["chunk_id"], 0) + 1 / (RRF_K + rank)

    for rank, chunk_id in enumerate(bm25_chunk_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (RRF_K + rank)

    top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:final_k]

    return [
        {
            "chunk_id": cid,
            "embedding": chunks_by_id[cid]["embedding"],
            "chunk_type": chunks_by_id[cid]["chunk_type"],
        }
        for cid in top_ids
    ]

## Run: baseline vs. MMR vs. hybrid, for every question

For each question: retrieve one pool of 15 vector candidates (shared by baseline and MMR) plus a separate BM25 top-15 (used only by hybrid), then produce three final sets — plain top-5, MMR-reranked top-5, and RRF-fused hybrid top-5 — scored the same way for a fair three-way comparison.


In [9]:
comparison_records = []

for item in tqdm(eval_set, desc="comparing baseline vs MMR vs hybrid"):
    candidates = get_candidates(item["question"])
    bm25_chunk_ids = get_bm25_candidates(item["question"])

    plain_top_k = candidates[:FINAL_K]
    mmr_top_k = mmr_rerank(candidates, k=FINAL_K)
    hybrid_top_k = rrf_fuse(candidates, bm25_chunk_ids, final_k=FINAL_K)

    for label, selected in [("baseline", plain_top_k), ("mmr", mmr_top_k), ("hybrid", hybrid_top_k)]:
        selected_ids = [c["chunk_id"] for c in selected]
        comparison_records.append({
            "question": item["question"],
            "expected_chunk_id": item["expected_chunk_id"],
            "method": label,
            "hit_at_k": item["expected_chunk_id"] in selected_ids,
            "avg_pairwise_similarity": avg_pairwise_similarity(selected),
            "distinct_chunk_types": len(set(c["chunk_type"] for c in selected)),
        })

comparison_df = pd.DataFrame(comparison_records)
comparison_df.head(15)

I0907 14:59:20.016725 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016796 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016799 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016800 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016801 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016803 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016805 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016806 61791516 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
I0907 14:59:20.016807 61791516 ev_poll_posix.cc:593] FD from fork parent still i

,question,expected_chunk_id,method,hit_at_k,avg_pairwise_similarity,distinct_chunk_types
0,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,baseline,True,0.702295,1
1,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,mmr,True,0.673037,1
2,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,hybrid,True,0.721874,1
3,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,baseline,True,0.693610,1
4,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,mmr,True,0.657174,1
5,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,hybrid,True,0.699688,1
6,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,baseline,True,0.574945,1
7,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,mmr,True,0.516539,1
8,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,hybrid,True,0.593873,1
9,What is the title of the study published by Po...,dc592958e93b4406881c295369e45fc6,baseline,True,0.723104,1


## Aggregate: three-way comparison

`baseline` vs `mmr` vs `hybrid` on the same three metrics. Things worth checking: does `hybrid` recover any of the recall MMR gave up (BM25's exact keyword matching can catch cases dense embeddings miss — like the bibliography-style question from Stage 5's diagnosis)? Does it also improve diversity, or is that specific to MMR's design?


In [10]:
summary = comparison_df.groupby("method").agg(
    recall_at_k=("hit_at_k", "mean"),
    avg_pairwise_similarity=("avg_pairwise_similarity", "mean"),
    avg_distinct_chunk_types=("distinct_chunk_types", "mean"),
)

display(summary)

,recall_at_k,avg_pairwise_similarity,avg_distinct_chunk_types
method,,,
baseline,0.80,0.702700,1.35
hybrid,0.90,0.699189,1.30
mmr,0.75,0.640399,1.45


In [11]:
def save_jsonl(units: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for unit in units:
            f.write(json.dumps(unit) + "\n")


save_jsonl(comparison_records, LOGS_DIR / "stage6_reranking_comparison.jsonl")
print(f"saved {len(comparison_records)} comparison records to {LOGS_DIR / 'stage6_reranking_comparison.jsonl'}")

saved 60 comparison records to /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/data/logs/stage6_reranking_comparison.jsonl


## Next: Stage 7 — Prompt template + LLM generation

With three reranking strategies now measured, Stage 7 picks one to carry forward as the final context-selection step before generation, builds the actual prompt template (separating text/table/image context clearly), and generates real answers using GPT-4o over the reranked chunks for each eval question — followed by Stage 8's RAGAS evaluation of those generated answers.
